In [1]:
import numpy as np
import pandas as pd

#### Concatenation

##### Concatenation is simply "pasting" two DataFrames together by columns or rows. Both DataFrames have to be in the same format

In [2]:
data_one = {'A': ['A0', 'A1', 'A2', 'A3'], 'B': ['B0', 'B1', 'B2', 'B3']}

In [3]:
data_two = {'C': ['C0', 'C1', 'C2', 'C3'], 'D': ['D0', 'D1', 'D2', 'D3']}

In [4]:
one = pd.DataFrame(data_one)

In [5]:
two = pd.DataFrame(data_two)

In [6]:
one

,A,B
0,A0,B0
1,A1,B1
2,A2,B2
3,A3,B3


In [7]:
two

,C,D
0,C0,D0
1,C1,D1
2,C2,D2
3,C3,D3


In [8]:
# to specify the concatenation along the columns we need to add axis=1
pd.concat([one, two], axis=1)

,A,B,C,D
0,A0,B0,C0,D0
1,A1,B1,C1,D1
2,A2,B2,C2,D2
3,A3,B3,C3,D3


In [9]:
# Concatenating along the rows
pd.concat([one, two], axis=0)

,A,B,C,D
0,A0,B0,NaN,NaN
1,A1,B1,NaN,NaN
2,A2,B2,NaN,NaN
3,A3,B3,NaN,NaN
0,NaN,NaN,C0,D0
1,NaN,NaN,C1,D1
2,NaN,NaN,C2,D2
3,NaN,NaN,C3,D3


##### Pandas automatially fills the table with null values because you cannot have two items in the same position, namely A0 and C0

In [10]:
# Changing the columns of the second dataset to have the values on the same column
two.columns = one.columns

In [11]:
two

,A,B
0,C0,D0
1,C1,D1
2,C2,D2
3,C3,D3


In [12]:
# Now, concatenate by the rows
pd.concat([one, two], axis=0)

,A,B
0,A0,B0
1,A1,B1
2,A2,B2
3,A3,B3
0,C0,D0
1,C1,D1
2,C2,D2
3,C3,D3


In [13]:
# To change the index to follow the right sequence
mydf = pd.concat([one, two], axis=0)

In [14]:
mydf.index = range(len(mydf))

In [15]:
mydf

,A,B
0,A0,B0
1,A1,B1
2,A2,B2
3,A3,B3
4,C0,D0
5,C1,D1
6,C2,D2
7,C3,D3


#### Merge - Inner Merge

##### When DataFrames are not in the exact same order or format
The .merge() method takes in a key argument labeled 'how'
There are 3 main ways of merging tables together using the 'how' parameter:
- Inner
- Outer
- Left or Right

In [16]:
registrations = pd.DataFrame({'reg_id':[1,2,3,4],'name':['Andrew','Bobo','Claire','David']})
logins = pd.DataFrame({'log_id':[1,2,3,4],'name':['Xavier','Andrew','Yolanda','Bobo']})

In [18]:
registrations

,reg_id,name
0,1,Andrew
1,2,Bobo
2,3,Claire
3,4,David


In [19]:
logins

,log_id,name
0,1,Xavier
1,2,Andrew
2,3,Yolanda
3,4,Bobo


In [21]:
# To get access to the index, we can reset index to get it as a column
# help(pd.merge)

In [22]:
pd.merge(registrations, logins, how='inner', on='name')

,reg_id,name,log_id
0,1,Andrew,2
1,2,Bobo,4


In [23]:
# For the inner merge, the table order doesn't really matter

##### Left and Right merge

In [24]:
# The order of the tables passed in as arguments matters here
# The left merge takes everything in the left table along with the common elements in both the left and right tables.
pd.merge(registrations, logins, how='left', on='name')

,reg_id,name,log_id
0,1,Andrew,2.0
1,2,Bobo,4.0
2,3,Claire,NaN
3,4,David,NaN


In [25]:
# We can replace the null values with some other value
pd.merge(registrations, logins, how='right', on='name')

,reg_id,name,log_id
0,NaN,Xavier,1
1,1.0,Andrew,2
2,NaN,Yolanda,3
3,2.0,Bobo,4


#### Outer Merge

##### An outer merge grabs everything from both tables. The order of placement for the tables doesn't matter in outer merge

In [26]:
pd.merge(registrations, logins, how='outer', on='name')

,reg_id,name,log_id
0,1.0,Andrew,2.0
1,2.0,Bobo,4.0
2,3.0,Claire,NaN
3,4.0,David,NaN
4,NaN,Xavier,1.0
5,NaN,Yolanda,3.0


#### Joining on an index instead of a column

In [27]:
registrations = registrations.set_index('name')

In [28]:
registrations

,reg_id
name,
Andrew,1
Bobo,2
Claire,3
David,4


In [29]:
logins

,log_id,name
0,1,Xavier
1,2,Andrew
2,3,Yolanda
3,4,Bobo


In [30]:
# We would have to specify that we want to join on the registraions index and the logins name column
pd.merge(registrations, logins, how='inner', left_index=True, right_on='name')

,reg_id,log_id,name
1,1,2,Andrew
3,2,4,Bobo


##### When dealing with different key-column names in the joined tables

In [31]:
# Reset the index
registrations = registrations.reset_index()
registrations

,name,reg_id
0,Andrew,1
1,Bobo,2
2,Claire,3
3,David,4


In [32]:
# Change the column names
registrations.columns = ['reg_name', 'reg_id']

In [33]:
registrations

,reg_name,reg_id
0,Andrew,1
1,Bobo,2
2,Claire,3
3,David,4


In [35]:
results = pd.merge(registrations, logins, how='inner', left_on='reg_name', right_on='name')

In [37]:
results

,reg_name,reg_id,log_id,name
0,Andrew,1,2,Andrew
1,Bobo,2,4,Bobo


In [39]:
# Get rid of the duplicate data
results.drop('reg_name', axis=1)

,reg_id,log_id,name
0,1,2,Andrew
1,2,4,Bobo


##### Add or tag duplicate columns

In [40]:
# Rename the columns
registrations.columns = ['name', 'id']

In [41]:
logins.columns = ['id', 'name']

In [42]:
registrations

,name,id
0,Andrew,1
1,Bobo,2
2,Claire,3
3,David,4


In [43]:
logins

,id,name
0,1,Xavier
1,2,Andrew
2,3,Yolanda
3,4,Bobo


In [44]:
pd.merge(registrations, logins, how='inner', on='name')

,name,id_x,id_y
0,Andrew,1,2
1,Bobo,2,4


In [45]:
# Specift the suffixes
pd.merge(registrations, logins, how='inner', on='name', suffixes=('_reg', '_log'))

,name,id_reg,id_log
0,Andrew,1,2
1,Bobo,2,4
